In [ ]:
import pickle
import pandas as pd

# Load data
with open("simulation_data.pkl", "rb") as f:
    df = pickle.load(f)

# Filter to plant agents only
plant_df = df[df["Agent Type"] == "plant"].reset_index()

# Keep only the last timestep per unique plant
latest_plants = (
    plant_df.sort_values("Step")
    .groupby("unique_id")
    .last()
)

# Now construct a clean summary DataFrame
summary_df = pd.DataFrame({
    "plant_id": latest_plants.index,
    "pos": latest_plants["Bee Positions"],
    "visit_timestamp": latest_plants["Plant visit time"],  # ✅ corrected
    "visit_count": latest_plants["Plant visit count"]
}).reset_index(drop=True)

summary_df.head()


In [ ]:
# Step 1: Split x and y from the 'pos' tuple into separate columns
summary_df[['x', 'y']] = summary_df['pos'].apply(pd.Series)

# Optional: round to avoid floating point mismatches
summary_df['y'] = summary_df['y'].round(2)

# Step 2: Now filter based on y
df_row1 = summary_df[summary_df['y'] == 2.85].reset_index(drop=True)
df_row2 = summary_df[summary_df['y'] == 5.70].reset_index(drop=True)
df_row3 = summary_df[summary_df['y'] == 8.55].reset_index(drop=True)

In [ ]:
def plot_visitation_by_x(summary_df, timestep_start, duration=60):
    """
    Plots number of bee visits at each x-coordinate from timestep_start to timestep_start + duration.

    Parameters:
    - summary_df: DataFrame with columns:
        - 'pos' (tuple of x, y)
        - 'visit_timestamp' (list of time steps)
    - timestep_start: Starting timestep
    - duration: Time window (default = 60)
    """

    # Extract x from pos
    summary_df['x'] = summary_df['pos'].apply(lambda pos: pos[0])

    # Count number of visits within the time window
    def count_visits_in_window(timestamps):
        return sum(timestep_start <= t < timestep_start + duration for t in timestamps)

    summary_df['visits_in_window'] = summary_df['visit_timestamp'].apply(count_visits_in_window)

    # Aggregate by x-coordinate
    visit_by_x = summary_df.groupby('x')['visits_in_window'].sum().reset_index()

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(visit_by_x['x'], visit_by_x['visits_in_window'], marker='o', linestyle='-')
    plt.xlabel('X Coordinate (Tunnel Length)')
    plt.ylabel('Number of Visits')
    plt.title(f'Bee Visitation from Step {timestep_start} to {timestep_start + duration}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_visitation_by_x(summary_df, timestep_start=120)
